In [5]:
!pip install nnetsauce openml joblib category_encoders --upgrade --no-cache-dir

# 1 - Classification

In [6]:
# -*- coding: utf-8 -*-
"""2024_05_19_openml_cc18_fixed.py

Fixed version of the OpenML CC-18 benchmark script.

Leakage fixes applied:
  1. train_test_split is called BEFORE any preprocessing.
  2. HashingEncoder is fit on X_train only, then transform-only on X_test.
  3. NaN medians are computed from X_train only, then applied to X_test.
  4. Feature-selection RandomForest is fit on X_train only; the same column
     indices are used to slice X_test (no refit).
  5. Row subsampling is applied to the train set only.

Saved artifact: 2026-05-06-openml-cc18-splits-fixed.pkl
  A dict keyed by dataset name, each value containing:
    {
      "task_id": int,
      "dataset": (X_train, X_test, y_train, y_test)   # all np.ndarray
    }
"""

import warnings
warnings.filterwarnings("ignore")

import category_encoders as ce
import joblib
import nnetsauce as ns
import numpy as np
import openml

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from time import time
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------
NROWS = 1000   # max rows kept in the TRAIN set after subsampling
NCOLS = 10     # max features kept after feature selection


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def impute_with_medians(arr: np.ndarray, medians: np.ndarray) -> np.ndarray:
    """Replace NaN values in arr using a pre-computed median vector."""
    arr = arr.copy()
    for col_idx in range(arr.shape[1]):
        nan_mask = np.isnan(arr[:, col_idx])
        if nan_mask.any():
            arr[nan_mask, col_idx] = medians[col_idx]
    return arr


# ---------------------------------------------------------------------------
# Leakage-free preprocessing + split
# ---------------------------------------------------------------------------
def preprocess_and_split(X_raw, y_raw):
    """
    Preprocess one OpenML task and return a clean train/test split.

    Order of operations (all fitting on train data only):
      1.  LabelEncode y
      2.  train_test_split on raw X, encoded y
      3.  HashingEncoder  — fit on X_train, transform X_test
      4.  NaN imputation  — medians from X_train, applied to both splits
      5.  Feature selection (RF importance) — fit on X_train, slice both splits
      6.  Row subsampling — train set only, test set untouched
    """
    print(f"  raw shapes  X={X_raw.shape}  y={np.asarray(y_raw).shape}")

    # 1. Encode labels
    le = LabelEncoder()
    y = le.fit_transform(np.asarray(y_raw)).astype(np.uint8)

    # 2. ✅ Split BEFORE any fitting on features
    X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. ✅ Categorical encoding — fit on train only
    encoder = ce.HashingEncoder(return_df=False)
    X_train = np.asarray(encoder.fit_transform(X_tr_raw, y_train)).astype(np.float32)
    X_test  = np.asarray(encoder.transform(X_te_raw)).astype(np.float32)

    # 4. ✅ NaN imputation — medians from train only
    train_medians = np.nanmedian(X_train, axis=0)
    X_train = impute_with_medians(X_train, train_medians)
    X_test  = impute_with_medians(X_test,  train_medians)

    # 5. ✅ Feature selection — RF fit on train only
    if X_train.shape[1] > NCOLS:
        print(f"  selecting top {NCOLS} / {X_train.shape[1]} features ...")
        rf_sel = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        rf_sel.fit(X_train, y_train)                          # train only!
        top_cols = np.argsort(rf_sel.feature_importances_)[::-1][:NCOLS]
        X_train  = X_train[:, top_cols]
        X_test   = X_test[:,  top_cols]                       # same indices, no refit
        print(f"  → kept columns: {top_cols}")

    # 6. ✅ Row subsampling — train set only
    if X_train.shape[0] > NROWS:
        print(f"  subsampling train from {X_train.shape[0]} → {NROWS} rows ...")
        t0  = time()
        sub = ns.SubSampler(
            y=y_train.ravel().astype(np.uint8),
            n_samples=NROWS, seed=123, n_jobs=-1
        )
        idx = sub.subsample().ravel()
        X_train = X_train[idx, :]
        y_train = y_train[idx]
        print(f"  subsampled in {time()-t0:.2f}s")

    y_train = y_train.ravel().astype(np.uint8)
    y_test  = y_test.ravel().astype(np.uint8)

    print(f"  final  train={X_train.shape}  test={X_test.shape}")
    return X_train, X_test, y_train, y_test


# ---------------------------------------------------------------------------
# 1 - Load and preprocess all CC-18 tasks
# ---------------------------------------------------------------------------
benchmark_suite = openml.study.get_suite('OpenML-CC18')
task_ids        = benchmark_suite.tasks
n_tasks         = len(task_ids)
print(f"Total tasks in CC-18: {n_tasks}\n")

splits = {}

overall_start = time()
for idx, task_id in tqdm(enumerate(task_ids), total=n_tasks):
    try:
        print(f"\n──── task #{idx+1}/{n_tasks}  (task_id={task_id}) ────")
        t0 = time()

        task      = openml.tasks.get_task(task_id)
        task_name = task.get_dataset().name
        X_raw, y_raw = task.get_X_and_y()

        X_train, X_test, y_train, y_test = preprocess_and_split(X_raw, y_raw)

        splits[task_name] = {
            "task_id": task_id,
            "dataset": (X_train, X_test, y_train, y_test),
        }
        print(f"  ✓ done in {time()-t0:.2f}s")

    except Exception as exc:
        print(f"  ✗ skipped — {exc}")
        continue

print(f"\nAll tasks processed in {time()-overall_start:.1f}s")
print(f"Successful datasets: {len(splits)} / {n_tasks}")


# ---------------------------------------------------------------------------
# 2 - Sanity check: RandomForest accuracy on each split
# ---------------------------------------------------------------------------
print("\n\n=== Sanity check: RandomForest accuracy per dataset ===\n")
clf = RandomForestClassifier(n_jobs=-1, random_state=42)

for i, (name, value) in enumerate(splits.items()):
    X_train, X_test, y_train, y_test = value["dataset"]
    t0  = time()
    acc = clf.fit(X_train, y_train).score(X_test, y_test)
    print(f"[{i+1:>3}/{len(splits)}]  {name:<40}  acc={acc:.4f}  ({time()-t0:.2f}s)")


# ---------------------------------------------------------------------------
# 3 - Save to pickle
# ---------------------------------------------------------------------------
OUTPUT_PATH = "2026-05-06-openml-cc18-splits-fixed.pkl"
joblib.dump(splits, OUTPUT_PATH)
print(f"\n✅ Saved {len(splits)} datasets to '{OUTPUT_PATH}'")
print("   Structure: dict[name] = {'task_id': int, 'dataset': (X_train, X_test, y_train, y_test)}")

Total tasks in CC-18: 72



  0%|          | 0/72 [00:00<?, ?it/s]


──── task #1/72  (task_id=3) ────
  raw shapes  X=(3196, 36)  y=(3196,)
  selecting top 10 / 36 features ...
  → kept columns: [20 32  9 14  5 34 31  6  7  0]
  subsampling train from 2556 → 1000 rows ...


  1%|▏         | 1/72 [00:01<01:11,  1.00s/it]

  subsampled in 0.74s
  final  train=(999, 10)  test=(640, 10)
  ✓ done in 1.00s

──── task #2/72  (task_id=6) ────
  raw shapes  X=(20000, 16)  y=(20000,)
  selecting top 10 / 16 features ...


...

 99%|█████████▊| 71/72 [06:37<00:31, 31.08s/it]

  → kept columns: [ 84  89  92 104  82  99  88  87  95  85]
  subsampling train from 2548 → 1000 rows ...
  subsampled in 0.02s
  final  train=(999, 10)  test=(638, 10)
  ✓ done in 0.59s

──── task #72/72  (task_id=167141) ────
  raw shapes  X=(5000, 20)  y=(5000,)
  selecting top 10 / 20 features ...


100%|██████████| 72/72 [06:38<00:00,  5.53s/it]

  → kept columns: [ 7  9 19  4 10 12 17 18 13 15]
  subsampling train from 4000 → 1000 rows ...
  subsampled in 0.03s
  final  train=(999, 10)  test=(1000, 10)
  ✓ done in 0.61s

All tasks processed in 398.3s
Successful datasets: 71 / 72


=== Sanity check: RandomForest accuracy per dataset ===



[  1/71]  kr-vs-kp                                  acc=0.9797  (0.37s)
[  2/71]  letter                                    acc=0.8020  (0.44s)
[  3/71]  balance-scale                             acc=0.8240  (0.31s)
[  4/71]  mfeat-factors                             acc=0.8950  (0.43s)
[  5/71]  mfeat-fourier                             acc=0.8325  (0.48s)
[  6/71]  breast-w                                  acc=0.9500  (0.35s)
[  7/71]  mfeat-karhunen                            acc=0.8975  (0.71s)
[  8/71]  mfeat-morphological                       acc=0.6975  (0.53s)
[  9/71]  mfeat-zernike                             acc=0.7000  (0.81s)
[ 10/71]  cmc                                       acc=0.5017  (0.48s)
[ 11/71]  optdigits                                 acc=0.8763  (0.55s)
[ 12/71]  credit-approval                           acc=0.8406  (0.40s)
[ 13/71]  credit-g                                  acc=0.7850  (0.37s)
[ 14/71]  pendigits                                 acc=0.9518  

# 2 - Regression

In [7]:
# -*- coding: utf-8 -*-
"""openml_regression_benchmark.py

Leakage-free OpenML regression benchmark script.
Mirrors the structure of 2024_05_19_openml_cc18_fixed.py (classification)
but targets OpenML-CTR23 (suite ID=353), the curated tabular regression
benchmark with 35 datasets.

NOTE: get_suite() requires the INTEGER id (353), not the string alias
      "OpenML-CTR23" which the REST API does not resolve.

Leakage fixes applied:
  1. train_test_split is called BEFORE any preprocessing.
  2. HashingEncoder is fit on X_train only, then transform-only on X_test.
  3. NaN medians are computed from X_train only, then applied to X_test.
  4. Feature-selection RandomForest is fit on X_train only; the same column
     indices are used to slice X_test (no refit).
  5. Row subsampling is applied to the train set only.
  6. Target y is log1p-transformed (if all-positive and large-range) —
     decision made from full y before splitting (metadata only, no stat
     leakage); inverse-transformed for evaluation.

Saved artifact: 2026-05-06-openml-regression-splits.pkl
  A dict keyed by dataset name, each value containing:
    {
      "task_id":         int,
      "dataset":         (X_train, X_test, y_train, y_test),  # np.ndarray
      "log_transformed": bool
    }
"""

import warnings
warnings.filterwarnings("ignore")

import category_encoders as ce
import joblib
import numpy as np
import openml

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from time import time
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------
NROWS    = 1000  # max rows kept in the TRAIN set after subsampling
NCOLS    = 10    # max features kept after feature selection
SUITE_ID = 353   # OpenML-CTR23  (integer id — the string alias does not resolve)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def impute_with_medians(arr: np.ndarray, medians: np.ndarray) -> np.ndarray:
    """Replace NaN values in arr using a pre-computed median vector."""
    arr = arr.copy()
    for col_idx in range(arr.shape[1]):
        nan_mask = np.isnan(arr[:, col_idx])
        if nan_mask.any():
            arr[nan_mask, col_idx] = medians[col_idx]
    return arr


# ---------------------------------------------------------------------------
# Leakage-free preprocessing + split
# ---------------------------------------------------------------------------
def preprocess_and_split(X_raw, y_raw):
    """
    Preprocess one OpenML regression task and return a clean train/test split.

    Order of operations (all fitting on train data only):
      1.  Cast y to float64; optional log1p-transform if y > 0 everywhere
      2.  train_test_split on raw X and y
      3.  HashingEncoder  — fit on X_train, transform X_test
      4.  NaN imputation  — medians from X_train, applied to both splits
      5.  Feature selection (RF importance) — fit on X_train, slice both splits
      6.  Row subsampling — train set only, test set untouched

    Returns
    -------
    X_train, X_test : np.ndarray (float32)
    y_train, y_test : np.ndarray (float64)  — in log1p space if transformed
    log_transformed : bool
    """
    y_raw = np.asarray(y_raw, dtype=np.float64).ravel()
    print(f"  raw shapes  X={X_raw.shape}  y={y_raw.shape}")

    # 1. Optional log1p transform.
    # Decision is a metadata choice on the full y (no statistics leak into test).
    log_transformed = bool(np.all(y_raw > 0) and np.max(y_raw) > 100)
    if log_transformed:
        y_raw = np.log1p(y_raw)
        print("  log1p-transforming y")

    # 2. ✅ Split BEFORE any fitting on features
    X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
        X_raw, y_raw, test_size=0.2, random_state=42
    )

    # 3. ✅ Categorical encoding — fit on train only
    encoder = ce.HashingEncoder(return_df=False)
    X_train = np.asarray(encoder.fit_transform(X_tr_raw)).astype(np.float32)
    X_test  = np.asarray(encoder.transform(X_te_raw)).astype(np.float32)

    # 4. ✅ NaN imputation — medians from train only
    train_medians = np.nanmedian(X_train, axis=0)
    X_train = impute_with_medians(X_train, train_medians)
    X_test  = impute_with_medians(X_test,  train_medians)

    # 5. ✅ Feature selection — RF fit on train only
    if X_train.shape[1] > NCOLS:
        print(f"  selecting top {NCOLS} / {X_train.shape[1]} features ...")
        rf_sel = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
        rf_sel.fit(X_train, y_train)                          # train only!
        top_cols = np.argsort(rf_sel.feature_importances_)[::-1][:NCOLS]
        X_train  = X_train[:, top_cols]
        X_test   = X_test[:,  top_cols]                       # same indices, no refit
        print(f"  → kept columns: {top_cols}")

    # 6. ✅ Row subsampling — train set only
    if X_train.shape[0] > NROWS:
        print(f"  subsampling train from {X_train.shape[0]} → {NROWS} rows ...")
        t0  = time()
        rng = np.random.default_rng(123)
        idx = rng.choice(X_train.shape[0], size=NROWS, replace=False)
        X_train = X_train[idx, :]
        y_train = y_train[idx]
        print(f"  subsampled in {time()-t0:.2f}s")

    y_train = y_train.ravel().astype(np.float64)
    y_test  = y_test.ravel().astype(np.float64)

    print(f"  final  train={X_train.shape}  test={X_test.shape}")
    return X_train, X_test, y_train, y_test, log_transformed


# ---------------------------------------------------------------------------
# 1 - Load and preprocess all CTR-23 regression tasks
# ---------------------------------------------------------------------------
benchmark_suite = openml.study.get_suite(SUITE_ID)   # ← integer id, not string alias
task_ids        = benchmark_suite.tasks
n_tasks         = len(task_ids)
print(f"Total tasks in OpenML-CTR23 (suite {SUITE_ID}): {n_tasks}\n")

splits = {}

overall_start = time()
for idx, task_id in tqdm(enumerate(task_ids), total=n_tasks):
    try:
        print(f"\n──── task #{idx+1}/{n_tasks}  (task_id={task_id}) ────")
        t0 = time()

        task      = openml.tasks.get_task(task_id)
        task_name = task.get_dataset().name
        X_raw, y_raw = task.get_X_and_y()

        # Skip tasks whose target is non-numeric or entirely missing
        y_arr = np.asarray(y_raw, dtype=float)
        if np.all(np.isnan(y_arr)):
            print("  ✗ skipped — target is all NaN")
            continue

        X_train, X_test, y_train, y_test, log_transformed = preprocess_and_split(
            X_raw, y_arr
        )

        splits[task_name] = {
            "task_id":         task_id,
            "dataset":         (X_train, X_test, y_train, y_test),
            "log_transformed": log_transformed,
        }
        print(f"  ✓ done in {time()-t0:.2f}s")

    except Exception as exc:
        print(f"  ✗ skipped — {exc}")
        continue

print(f"\nAll tasks processed in {time()-overall_start:.1f}s")
print(f"Successful datasets: {len(splits)} / {n_tasks}")


# ---------------------------------------------------------------------------
# 2 - Sanity check: RandomForest R² and MAE on each split
# ---------------------------------------------------------------------------
print("\n\n=== Sanity check: RandomForest R² / MAE per dataset ===\n")
reg = RandomForestRegressor(n_jobs=-1, random_state=42)

for i, (name, value) in enumerate(splits.items()):
    X_train, X_test, y_train, y_test = value["dataset"]
    log_transformed                  = value["log_transformed"]

    t0  = time()
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)

    # Report metrics in original (non-log) scale
    if log_transformed:
        y_pred_orig = np.expm1(y_pred)
        y_true_orig = np.expm1(y_test)
    else:
        y_pred_orig = y_pred
        y_true_orig = y_test

    r2  = r2_score(y_true_orig, y_pred_orig)
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    log_tag = " [log1p]" if log_transformed else ""
    print(
        f"[{i+1:>3}/{len(splits)}]  {name:<40}  "
        f"R²={r2:+.4f}  MAE={mae:.4f}  ({time()-t0:.2f}s){log_tag}"
    )


# ---------------------------------------------------------------------------
# 3 - Save to pickle
# ---------------------------------------------------------------------------
OUTPUT_PATH = "2026-05-06-openml-regression-splits.pkl"
joblib.dump(splits, OUTPUT_PATH)
print(f"\n✅ Saved {len(splits)} datasets to '{OUTPUT_PATH}'")
print(
    "   Structure: dict[name] = {\n"
    "     'task_id': int,\n"
    "     'dataset': (X_train, X_test, y_train, y_test),\n"
    "     'log_transformed': bool\n"
    "   }"
)

Total tasks in OpenML-CTR23 (suite 353): 35



 14%|█▍        | 5/35 [00:00<00:00, 36.88it/s]


──── task #1/35  (task_id=361234) ────
  raw shapes  X=(4177, 8)  y=(4177,)
  subsampling train from 3341 → 1000 rows ...
  subsampled in 0.00s
  final  train=(1000, 8)  test=(836, 8)
  ✓ done in 0.02s

──── task #2/35  (task_id=361235) ────
  raw shapes  X=(1503, 5)  y=(1503,)
  log1p-transforming y
  subsampling train from 1202 → 1000 rows ...
  subsampled in 0.00s
  final  train=(1000, 5)  test=(301, 5)
  ✓ done in 0.01s

──── task #3/35  (task_id=361236) ────
  raw shapes  X=(2043, 7)  y=(2043,)
  log1p-transforming y
  subsampling train from 1634 → 1000 rows ...
  subsampled in 0.00s
  final  train=(1000, 7)  test=(409, 7)
  ✓ done in 0.02s

──── task #4/35  (task_id=361237) ────
  raw shapes  X=(1030, 8)  y=(1030,)
  final  train=(824, 8)  test=(206, 8)
  ✓ done in 0.01s

──── task #5/35  (task_id=361241) ────
  raw shapes  X=(45730, 9)  y=(45730,)
  subsampling train from 36584 → 1000 rows ...
  subsampled in 0.00s
  final  train=(1000, 9)  test=(9146, 9)
  ✓ done in 0.07s

───

 17%|█▋        | 6/35 [01:10<07:36, 15.76s/it]

  → kept columns: [67 31 64  2 50 27 44 62 69 74]
  subsampling train from 17010 → 1000 rows ...
  subsampled in 0.00s
  final  train=(1000, 10)  test=(4253, 10)
  ✓ done in 70.81s

──── task #7/35  (task_id=361243) ────
  raw shapes  X=(1059, 116)  y=(1059,)
  selecting top 10 / 116 features ...


...

 91%|█████████▏| 32/35 [07:34<00:11,  3.92s/it]

  → kept columns: [14 29  7  0 27 24 26  8  2 28]
  final  train=(519, 10)  test=(130, 10)
  ✓ done in 0.42s

──── task #33/35  (task_id=361621) ────
  raw shapes  X=(908, 6)  y=(908,)
  final  train=(726, 6)  test=(182, 6)
  ✓ done in 0.02s

──── task #34/35  (task_id=361622) ────
  raw shapes  X=(804, 17)  y=(804,)
  log1p-transforming y
  selecting top 10 / 17 features ...


100%|██████████| 35/35 [07:35<00:00, 13.01s/it]

  → kept columns: [ 1 10  7  0  9 16 12  8  6  5]
  final  train=(643, 10)  test=(161, 10)
  ✓ done in 0.31s

──── task #35/35  (task_id=361623) ────
  raw shapes  X=(3107, 6)  y=(3107,)
  subsampling train from 2485 → 1000 rows ...
  subsampled in 0.00s
  final  train=(1000, 6)  test=(622, 6)
  ✓ done in 0.02s

All tasks processed in 455.2s
Successful datasets: 34 / 35


=== Sanity check: RandomForest R² / MAE per dataset ===



[  1/34]  abalone                                   R²=+0.5253  MAE=1.6383  (1.02s)
[  2/34]  airfoil_self_noise                        R²=+0.9219  MAE=1.4554  (0.35s) [log1p]
[  3/34]  auction_verification                      R²=+0.9907  MAE=462.7063  (0.32s) [log1p]
[  4/34]  concrete_compressive_strength             R²=+0.8841  MAE=3.7516  (0.49s)
[  5/34]  physiochemical_protein                    R²=+0.3648  MAE=3.7462  (0.95s)
[  6/34]  superconductivity                         R²=+0.8141  MAE=9.1094  (0.88s) [log1p]
[  7/34]  geographical_origin_of_music              R²=+0.2236  MAE=12.4959  (0.85s)
[  8/34]  solar_flare                               R²=-0.0775  MAE=0.4521  (0.27s)
[  9/34]  naval_propulsion_plant                    R²=+0.9613  MAE=0.0017  (0.88s)
[ 10/34]  white_wine                                R²=+0.4010  MAE=0.5245  (0.68s)
[ 11/34]  red_wine                                  R²=+0.4875  MAE=0.4372  (0.64s)
[ 12/34]  grid_stability                         